# Funções e importação dos dados

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error
from collections import Counter
from tqdm.auto import tqdm


# Criar uma nova coluna para partidas, exibindo os resultados da partida
# 0 - Vitótia da casa
# 1 - Empate
# 2 - Derrota da casa
def resultado(row):
  if row["home_goals"] > row["away_goals"]:
    return 0      # vitória da casa
  elif row["home_goals"] < row["away_goals"]:
    return 2      # vitória visitante
  else:
    return 1      # empate

# Importação dos datasets do kaggle
partidas = pd.read_csv('teams_match_features.csv')

# Exibição inicial dos dados

In [3]:
# Exibição inicial do dataset
display(partidas)
print()

# Tamanho e algumas informações
print(partidas.shape)
partidas.info()
print()

# Número de valores nulos do dataset
partidas.isnull().sum().sort_values(ascending=False)

,home_elo,away_elo,elo_diff,home_avg_overall,home_max_overall,home_avg_attack,home_avg_defense,home_avg_pace,home_avg_shooting,home_avg_passing,...,away_form_win_rate,is_neutral,is_world_cup,is_continental,home_goals,away_goals,_home_team,_away_team,_date,_tournament
0,1500.000000,1500.000000,0.000000,74.285714,76,74.500000,74.250000,71.083333,69.500000,69.583333,...,0.330000,0,0,0,0,0,Scotland,England,1872-11-30,Friendly
1,1500.000000,1500.000000,0.000000,81.500000,86,82.250000,81.222222,73.538462,69.923077,71.307692,...,0.000000,0,0,0,4,2,England,Scotland,1873-03-08,Friendly
2,1484.000000,1516.000000,-32.000000,74.285714,76,74.500000,74.250000,71.083333,69.500000,69.583333,...,0.500000,0,0,0,2,1,Scotland,England,1874-03-07,Friendly
3,1498.530498,1501.469502,-2.939003,81.500000,86,82.250000,81.222222,73.538462,69.923077,71.307692,...,0.333333,0,0,0,2,2,England,Scotland,1875-03-06,Friendly
4,1501.334159,1498.665841,2.668317,74.285714,76,74.500000,74.250000,71.083333,69.500000,69.583333,...,0.250000,0,0,0,3,0,Scotland,England,1876-03-04,Friendly
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43359,1665.264832,1718.920373,-53.655541,71.000000,71,71.000000,71.000000,82.000000,65.000000,63.000000,...,0.200000,1,0,0,3,0,Jordan,Egypt,2025-12-09,Arab Cup
43360,1490.482978,1463.662216,26.820762,72.000000,72,72.000000,72.000000,74.000000,68.000000,52.000000,...,0.200000,1,0,0,3,1,Bahrain,Sudan,2025-12-09,Arab Cup
43361,1783.496877,1722.025083,61.471795,77.357143,86,78.571429,77.000000,71.000000,67.500000,74.142857,...,0.600000,1,0,0,2,0,Algeria,Iraq,2025-12-09,Arab Cup
43362,1702.726099,1548.049183,154.676916,71.928571,77,71.500000,71.900000,75.923077,54.846154,62.307692,...,0.200000,1,0,0,2,1,Saudi Arabia,Palestine,2025-12-11,Arab Cup



(43364, 35)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43364 entries, 0 to 43363
Data columns (total 35 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   home_elo            43364 non-null  float64
 1   away_elo            43364 non-null  float64
 2   elo_diff            43364 non-null  float64
 3   home_avg_overall    43364 non-null  float64
 4   home_max_overall    43364 non-null  int64  
 5   home_avg_attack     43364 non-null  float64
 6   home_avg_defense    43364 non-null  float64
 7   home_avg_pace       42667 non-null  float64
 8   home_avg_shooting   42667 non-null  float64
 9   home_avg_passing    42667 non-null  float64
 10  away_avg_overall    43364 non-null  float64
 11  away_max_overall    43364 non-null  int64  
 12  away_avg_attack     43364 non-null  float64
 13  away_avg_defense    43364 non-null  float64
 14  away_avg_pace       42563 non-null  float64
 15  away_avg_shooting   42563 non-null  floa

away_avg_pace         801
away_avg_passing      801
away_avg_shooting     801
home_avg_pace         697
home_avg_passing      697
home_avg_shooting     697
home_elo                0
elo_diff                0
away_elo                0
home_avg_defense        0
home_avg_overall        0
away_avg_overall        0
home_max_overall        0
home_avg_attack         0
away_max_overall        0
away_avg_defense        0
away_avg_attack         0
overall_diff            0
attack_diff             0
defense_diff            0
home_form_scored        0
home_form_conceded      0
home_form_win_rate      0
away_form_scored        0
away_form_conceded      0
away_form_win_rate      0
is_neutral              0
is_world_cup            0
is_continental          0
home_goals              0
away_goals              0
_home_team              0
_away_team              0
_date                   0
_tournament             0
dtype: int64

# Modelo de contgem dos gols

### Regressão de Poisson

In [4]:
y_casa = partidas['home_goals']
y_fora = partidas['away_goals']

features = [
    # Elo
    "home_elo",
    "away_elo",
    "elo_diff",

    # EA FC
    "overall_diff",
    "attack_diff",
    "defense_diff",

    # Forma
    "home_form_scored",
    "home_form_conceded",
    "home_form_win_rate",

    "away_form_scored",
    "away_form_conceded",
    "away_form_win_rate",

    # Local
    "is_neutral"
]

x = partidas[features]

x_train, x_test, y_casa_train, y_casa_test = train_test_split(
    x, y_casa,
    test_size = 0.2,
    random_state = 42
)

_, _, y_fora_train, y_fora_test = train_test_split(
    x, y_fora,
    test_size = 0.2,
    random_state = 42
)

modelo_casa = PoissonRegressor(alpha = 0.1)
modelo_casa.fit(x_train, y_casa_train)

modelo_fora = PoissonRegressor(alpha = 0.1)
modelo_fora.fit(x_train, y_fora_train)

pred_casa = modelo_casa.predict(x_test)
print(mean_absolute_error(y_casa_test, pred_casa))

pred_fora = modelo_fora.predict(x_test)
print(mean_absolute_error(y_fora_test, pred_fora))

1.2322493594126382
0.8308704188535838


/home/kevin/Área de Trabalho/trabalhodacopa2026/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/home/kevin/Área de Trabalho/trabalhodacopa2026/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: invalid value encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights


# Simulação dos jogos

### Criação do modelo

In [5]:
def simular_partida(features):
    X = pd.DataFrame([features])

    lambda_casa = modelo_casa.predict(X)[0]
    lambda_fora = modelo_fora.predict(X)[0]

    gols_casa = np.random.poisson(lambda_casa)
    gols_fora = np.random.poisson(lambda_fora)

    return gols_casa, gols_fora

In [6]:
# Teste da função, com o seguinte modelo de features (mesmas features usadas
# durante o treinamento)
features = {
    "home_elo": 2055,
    "away_elo": 1980,
    "elo_diff": 75,
    "overall_diff": 2.1,
    "attack_diff": 3.5,
    "defense_diff": 1.2,
    "home_form_scored": 2.4,
    "home_form_conceded": 0.6,
    "home_form_win_rate": 0.80,
    "away_form_scored": 1.9,
    "away_form_conceded": 0.8,
    "away_form_win_rate": 0.70,
    "is_neutral": 1
}

gols_casa, gols_fora = simular_partida(features)
print(gols_casa, gols_fora)

0 1


In [ ]:
resultado = []

for _ in range(10000):
  g1, g2 = simular_partida(features)
  resultado.append((g1, g2))
# Tempo de execução: 12 segundos

In [8]:
vitoria_casa = 0
empate = 0
vitoria_fora = 0

for g1, g2 in resultado:
  if g1 > g2:
    vitoria_casa += 1
  elif g2 > g1:
    vitoria_fora += 1
  else:
    empate += 1

valores_resultado = [
    ['Casa', vitoria_casa, vitoria_casa / 100],
    ['Empate', empate, empate / 100],
    ['Fora', vitoria_fora, vitoria_fora / 100]
]
resultado_data = pd.DataFrame(valores_resultado, columns = ['_', 'jogos_ganhos', 'percentual_de_vitoria'])
display(resultado_data)

,_,jogos_ganhos,percentual_de_vitoria
0,Casa,5751,57.51
1,Empate,2287,22.87
2,Fora,1962,19.62


### Simulação da copa

In [9]:
# Organização das datas do dataset
partidas["_date"] = pd.to_datetime(partidas["_date"])
partidas = partidas.sort_values("_date")


# Criação do datset de estatísticas de cada seleção
casa = partidas[[
    "_date",
    "_home_team",
    "home_elo",
    "home_avg_overall",
    "home_avg_attack",
    "home_avg_defense",
    "home_form_scored",
    "home_form_conceded",
    "home_form_win_rate"
]].rename(columns={
    "_home_team": "team",
    "home_elo": "elo",
    "home_avg_overall": "overall",
    "home_avg_attack": "attack",
    "home_avg_defense": "defense",
    "home_form_scored": "form_scored",
    "home_form_conceded": "form_conceded",
    "home_form_win_rate": "form_win_rate"
})

fora = partidas[[
    "_date",
    "_away_team",
    "away_elo",
    "away_avg_overall",
    "away_avg_attack",
    "away_avg_defense",
    "away_form_scored",
    "away_form_conceded",
    "away_form_win_rate"
]].rename(columns={
    "_away_team": "team",
    "away_elo": "elo",
    "away_avg_overall": "overall",
    "away_avg_attack": "attack",
    "away_avg_defense": "defense",
    "away_form_scored": "form_scored",
    "away_form_conceded": "form_conceded",
    "away_form_win_rate": "form_win_rate"
})

selecoes = pd.concat([casa, fora], ignore_index=True)
selecoes = (
    selecoes
    .sort_values("_date")
    .groupby("team", as_index=False)
    .last()
)

selecoes = selecoes.set_index('team')

# Infelizmente, não tem jogos do cabo verde no dataset ={
selecoes.loc["Cape Verde Islands"] = {
    "elo": 1680,
    "overall": 73.5,
    "attack": 73,
    "defense": 72,
    "form_scored": 1.45,
    "form_conceded": 0.95,
    "form_win_rate": 0.58
}

/tmp/ipykernel_11822/747240851.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  selecoes.loc["Cape Verde Islands"] = {


In [226]:
def criar_features(time_casa, time_fora):
  casa = selecoes.loc[time_casa]
  fora = selecoes.loc[time_fora]

  return pd.DataFrame([{
      'home_elo': casa['elo'],
      'away_elo': fora['elo'],
      'elo_diff': casa['elo'] - fora['elo'],

      'overall_diff': casa['overall'] - fora['overall'],
      'attack_diff': casa['attack'] - fora['attack'],
      'defense_diff': casa['defense'] - fora['defense'],

      'home_form_scored': casa['form_scored'],
      'home_form_conceded': casa['form_conceded'],
      'home_form_win_rate': casa['form_win_rate'],

      'away_form_scored': fora['form_scored'],
      'away_form_conceded': fora['form_conceded'],
      'away_form_win_rate': fora['form_win_rate'],

      'is_neutral': 1
  }])

def simular_partida(time1, time2):
  X = criar_features(time1, time2)

  lambda_casa = modelo_casa.predict(X)[0]
  lambda_fora = modelo_fora.predict(X)[0]

  gols_casa = np.random.poisson(lambda_casa)
  gols_fora = np.random.poisson(lambda_fora)

  if gols_casa > gols_fora:
    return time1
  if gols_fora > gols_casa:
    return time2

  # Prorrogação
  gols_casa += np.random.poisson(lambda_casa /3)
  gols_fora += np.random.poisson(lambda_fora / 3)

  if gols_casa > gols_fora:
    return time1
  if gols_fora > gols_casa:
    return time2

  # Pênaltis
  elo1 = selecoes.loc[time1, "elo"]
  elo2 = selecoes.loc[time2, "elo"]

  prob = elo1 / (elo1 + elo2)

  vencedor = time1 if np.random.rand() < prob else time2
  return vencedor

grupos = [
    ['Germany', 'Paraguay', 'France', 'Sweden'],
    ['South Africa', 'Canada', 'Netherlands', 'Morocco'],
    ['Portugal', 'Croatia', 'Spain', 'Austria'],
    ['United States', 'Bosnia and Herzegovina', 'Belgium', 'Senegal'],
    ['Brazil', 'Japan', "Ivory Coast", 'Norway'],
    ['Mexico', 'Ecuador', 'England', 'Congo'],
    ['Argentina', 'Cape Verde Islands', 'Australia', 'Egypt'],
    ['Switzerland', 'Algeria', 'Colombia', 'Ghana']
]

# Simulação da copa
def simular_copa(grupos):

    # Oitavas
    oitavas = []
    for chave in grupos:
        vencedor1 = simular_partida(chave[0], chave[1])
        vencedor2 = simular_partida(chave[2], chave[3])
        oitavas.append([vencedor1, vencedor2])

    # Quartas
    quartas = []
    for i in range(4):
        vencedor1 = simular_partida(oitavas[i * 2][0], oitavas[i * 2][1])
        vencedor2 = simular_partida(oitavas[i * 2 + 1][0], oitavas[i * 2 + 1][1])
        quartas.append([vencedor1, vencedor2])

    # Semifinais
    semis = []
    for i in range(2):
        vencedor1 = simular_partida(quartas[i * 2][0], quartas[i * 2][1])
        vencedor2 = simular_partida(quartas[i * 2 + 1][0], quartas[i * 2 + 1][1])
        semis.append([vencedor1, vencedor2])

    # Final
    finalista1 = simular_partida(semis[0][0], semis[0][1])
    finalista2 = simular_partida(semis[1][0], semis[1][1])

    campeao = simular_partida(finalista1, finalista2)

    return {
        "oitavas": oitavas,
        "quartas": quartas,
        "semis": semis,
        "finalistas": [finalista1, finalista2],
        "campeao": campeao
    }

### Simulação Monte Carlo

In [231]:
N = 10000

campeoes = Counter()
finalistas = Counter()
semifinalistas = Counter()
quartas_finalistas = Counter()

for _ in tqdm(range(N), desc="Simulando Copas"):
    resultado = simular_copa(grupos)
    campeoes[resultado["campeao"]] += 1

    for t in resultado["finalistas"]:
        finalistas[t] += 1

    for confronto in resultado["semis"]:
        for t in confronto:
            semifinalistas[t] += 1

    for confronto in resultado["quartas"]:
        for t in confronto:
            quartas_finalistas[t] += 1

times = sorted(set(campeoes) |
               set(finalistas) |
               set(semifinalistas) |
               set(quartas_finalistas))

resultado = pd.DataFrame(index=times)

resultado["Prob. Quartas (%)"] = [100 * quartas_finalistas[t] / N for t in times]
resultado["Prob. Semi (%)"] = [100 * semifinalistas[t] / N for t in times]
resultado["Prob. Final (%)"] = [100 * finalistas[t] / N for t in times]
resultado["Prob. Campeão (%)"] = [100 * campeoes[t] / N for t in times]

resultado = resultado.sort_values("Prob. Campeão (%)", ascending=False)
display(resultado)

Simulando Copas:   0%|          | 0/10000 [00:00<?, ?it/s]

,Prob. Quartas (%),Prob. Semi (%),Prob. Final (%),Prob. Campeão (%)
Germany,50.38,35.87,25.78,18.32
France,20.76,14.99,10.62,7.69
Portugal,50.73,36.45,10.33,7.62
Paraguay,20.07,14.38,10.29,7.36
Brazil,50.87,36.03,25.40,7.02
South Africa,49.99,13.99,9.88,6.85
Mexico,51.18,15.05,10.78,3.27
Netherlands,21.02,6.26,4.54,3.27
Argentina,51.47,37.24,10.53,3.19
Spain,20.45,14.76,4.41,3.15
